# Cellfix6 — candidate đóng băng, chờ xác nhận annotation validation

Source/prompt/contract/criteria và 89 đầu vào cellfix6 giữ nguyên hash đã kiểm.
Đính kèm bản nháp validation 30 ca từ 10 paper QASPER khác bộ development.
Nhãn do AI đề xuất, **chưa phải gold đã được người review xác nhận**.

Trước khi chạy GPU: mở `docs/validation/atomic-cellfix6-v1/REVIEW.html`, đọc context,
xác nhận riêng từng tiêu chí, ghi chú và xuất `annotation-review-cellfix6-v1.json`.
Cell 2 yêu cầu upload file này, kiểm hash/candidate/đủ 30 ca và tạo validation-lock.
Nếu annotation sai, sửa thành phiên bản dữ liệu mới trước inference; không bỏ gate.
Các giới hạn provenance trong form vẫn áp dụng sau khi được xác nhận.

Chọn runtime mới, GPU T4, chạy 5 cell. Cell 1 giải nén; cell 2 nhận review rồi dựng
môi trường; cell 3 chạy regression, ba validate-only và tải model đúng revision;
cell 4 chạy smoke12; cell 5 chỉ chạy probe18 nếu smoke đạt. Lỗi vẫn tải ZIP chẩn đoán.

Giữ Qwen/Qwen2.5-3B-Instruct aa8e72537993ba99e69dfaafa59ed015b17504d1,
NF4/float16/seed42; 8 call/ca, 2 attempt/stage, 1536 token/call, 180s/ca.
Không dùng kết quả validation để tune rồi gọi lại cùng bộ là độc lập.

**Notebook này chỉ chạy smoke12 và probe18.** Full39 và validation30 không tự chạy.
Probe mới cần review mới; sau đó chạy full39 dưới strict-identity-v1, rồi mới đo
validation đã khóa trên cùng candidate. Nhãn gốc cellfix5 21/39, false support=0
giữ nguyên để đối chiếu. Cellfix6 chưa có kết quả model/full39/validation hay E2E.


**1 — Nạp payload và kiểm hash**

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import base64, hashlib, io, json, os, signal, subprocess, sys, time, uuid, zipfile
from google.colab import files

smoke_passed = False
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
run_root = Path('/content')/('atomic-cellfix6-fresh-'+run_id)
root = run_root/'workspace'
out = run_root/('output-cellfix6-fresh-'+run_id)
root.mkdir(parents=True,exist_ok=False)
out.mkdir(exist_ok=False)
statuses = {}
env = os.environ.copy()
env.update(PYTHONPATH=str(root),HF_HOME=str(run_root/'hf'),
    HF_HUB_DISABLE_PROGRESS_BARS='1',TOKENIZERS_PARALLELISM='false')
env.pop('HF_HUB_OFFLINE',None)
env.pop('TRANSFORMERS_OFFLINE',None)
venv = run_root/'venv'
python = str(venv/'bin/python')
payload = '...'
payload_sha256 = '3d9d570f6bc7468f176c19952bbafe6212b609adf672fd3285a79f875ef9582a'
manifest = {'04_factchecker/checkpoints/factchecker_merged/chat_template.jinja': {'size': 2507, 'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f'}, '04_factchecker/checkpoints/factchecker_merged/config.json': {'size': 1548, 'sha256': '2e13d6edf2413d55422a2fe541d4fc851e17b01cc7a16601a107547687e7951d'}, '04_factchecker/checkpoints/factchecker_merged/generation_config.json': {'size': 242, 'sha256': 'dd1a52c6b9e837741be9dfa5c7fb8f0d8d9ed57e06d399a0fe52ec23d10ca066'}, '04_factchecker/checkpoints/factchecker_merged/tokenizer.json': {'size': 11421892, 'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8'}, '04_factchecker/checkpoints/factchecker_merged/tokenizer_config.json': {'size': 664, 'sha256': 'd9bf1d78a80bad5eb65b83c2608910e3be00a9d3250c8153a5a393ad1257e1d6'}, '04_factchecker/config.py': {'size': 2026, 'sha256': '2d9f99a6dee5a7a4b40278a552002d7d9737e060730bc9d5e48de4923693a584'}, '04_factchecker/eval_factchecker.py': {'size': 7759, 'sha256': '579907c751d107f9b1634511501b9dfd433ac9d583d551d989e1b95f6f7cbc0e'}, '04_factchecker/generate_training_data.py': {'size': 6255, 'sha256': 'e42e8abde6571f2257b2a05500b73395ec51c41407e4b35b872ef38684533e38'}, '04_factchecker/inference/__init__.py': {'size': 474, 'sha256': '11edee781c7d9592646acc9ebbca280c818fe561abb487995b079cbbe3ac6495'}, '04_factchecker/inference/action_recommender.py': {'size': 1846, 'sha256': '9e8d9350142122e573ed4ab758581b0076dc39d339566dd598c9f757fba9c37d'}, '04_factchecker/inference/assessment_rules.py': {'size': 8569, 'sha256': 'd18cbaf64ef60702489e2f71e8d62d7a8051ced3f08a9990af1666da1aa448e7'}, '04_factchecker/inference/atomic_decoding.py': {'size': 22689, 'sha256': 'f0ab78db4b1aafbdbd3b4cffde78ff945fdcd4b6d0f834e470a1497910b9359f'}, '04_factchecker/inference/atomic_verifier.py': {'size': 70818, 'sha256': '420a796430647881cfee16014f4be287e6e66aead9c91eb50f57e497e2e9a2da'}, '04_factchecker/inference/confidence.py': {'size': 3463, 'sha256': '307cdfb8bb9509a5189a31994d0aaa4c37f662e008f36dadaa54dcabeba0a09b'}, '04_factchecker/inference/evidence_screen.py': {'size': 5405, 'sha256': '23574b7320ac576989b283a58d3e81d58be001f1b1c286523d4f0baa04dfcab5'}, '04_factchecker/inference/judge.py': {'size': 5412, 'sha256': '818aa5a4dc7f8065e101264e59d57828cc00f60d1c105156e49c03f374f6dbb7'}, '04_factchecker/inference/numeric_verifier.py': {'size': 17099, 'sha256': 'aa403ae358512b470f1d111fb82a26d989bd8e277a417928fb9c39dc5b32334e'}, '04_factchecker/inference/verifier.py': {'size': 11971, 'sha256': '7eff9f8678cf1a4c270e59ada432bdf12a2fd6323c3d598fa34cc3513fd468f8'}, '04_factchecker/merge_lora_weights.py': {'size': 6730, 'sha256': 'bb237ec90020328f9cb2a849f75f4688e913bc41f35b661781ad113a4af78420'}, '04_factchecker/node.py': {'size': 7793, 'sha256': '577e62d2ce330d19bc4791641066d8d17b8dc4baebfc3dab455d593658afc48c'}, '04_factchecker/prompts.py': {'size': 7967, 'sha256': 'fe7be89a4ac6f789f9629f90c9710aeafc074806c066fa2451cb889121b6017e'}, '04_factchecker/train_factchecker_qlora.py': {'size': 21854, 'sha256': '99cee60ec35a3b60f9da41719891c3130ec5c64bf38e7e5bd4872382b13cc379'}, '04_factchecker/validate_labels.py': {'size': 3825, 'sha256': '1c70278ad4ab59659dade53b8f8eb6f4a1cf135d7ab957d685b248a5b1cd9877'}, 'requirements-atomic-verifier.txt': {'size': 131, 'sha256': 'b95eaa539ed6046f9261b09cdaa82017360568703d8af8d2bc6d8eed3c9af370'}, 'requirements-orchestrator.txt': {'size': 467, 'sha256': '31b5dac851eaaf005a151e9c071226aeafeb4679ebe4019d0368e68fbd0dacd4'}, 'research_rag/__init__.py': {'size': 134, 'sha256': 'cc7fea0e7e9fe57abaea1ca3e78e70ecaad9feb9819b088c50cdf26ba0d249e5'}, 'research_rag/adapters.py': {'size': 5305, 'sha256': '22537d4284338c66a1262d03dd3ce55dea4a95a2fa859ae78c638b66c45f147c'}, 'research_rag/controller.py': {'size': 7426, 'sha256': 'b97466aece81f27bd89d62f1fd0c61857a8a604b2b9681faf1850cdfb55ff43f'}, 'research_rag/evaluate.py': {'size': 10760, 'sha256': '48f5936f8ef3e3454ab7d2049d803be18d67134279fe14359e4c2ec0451fcf66'}, 'research_rag/evaluation/__init__.py': {'size': 66, 'sha256': 'edf15e92c3397d31441733113a0ea6bc45d805a6084ef354526ffce327d7dd5d'}, 'research_rag/evaluation/atomic_binding_adjudication_cellfix6.json': {'size': 2105, 'sha256': 'ad031bc547ea0af731b27604583c3deb7ce233a4e694a0f0341942124d0cbf84'}, 'research_rag/evaluation/atomic_verifier_contract_v8.json': {'size': 8468, 'sha256': 'd271a605783c4bd294e500c1488e56974d3b0759edc501b9e725515cf9a2edd6'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix1.json': {'size': 8995, 'sha256': '84e42bb71fe0d088adf3abe3ebd5660d3a2cdfe4f0d601e943cdaa04706c4cf5'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix2.json': {'size': 9302, 'sha256': '086fb6f181361ee64885c915b1cc67f636870b3b264a8017245fd566edf74574'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix3.json': {'size': 9978, 'sha256': '82feadb1b0cfb15490a825867ec3a2159cfa5253cc2fa79267ed9525907cf126'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix4.json': {'size': 10847, 'sha256': '80f1c75d8e91230a1d72a495f49b8dfcfa9bde57c2632b76d1a381624fdb6322'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix5.json': {'size': 12243, 'sha256': 'c37fda6dded8c7faa34dc7dbad4523944e07c92d20741832dc72069e0b7d0057'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix6.json': {'size': 13401, 'sha256': '699d07058de163d2344a787f8a9b328db21b1fae0ab22e9c84f236e87c32fb7a'}, 'research_rag/evaluation/atomic_verifier_criteria_v8.json': {'size': 14249, 'sha256': 'b9d77823b5c8d6d2a56d4053667b8b53f6e059a1f5edb9e7e3a7578a50f34071'}, 'research_rag/evaluation/atomic_verifier_logic_v3_baseline.json': {'size': 31343, 'sha256': '42f8fab1e7b5a9056538481113d6590d4e84d0b089989472afe09f9ccd16be85'}, 'research_rag/evaluation/blind_verifier_scope_regression_cases.jsonl': {'size': 16356, 'sha256': 'f349b1d6a50ca417c28c43d5fced1f064fc43ebdf362d70acfc574d7d37ed45c'}, 'research_rag/evaluation/compare_blind_verifier.py': {'size': 16716, 'sha256': 'ea14524f1f43c806975e0adbe0c0f24eecf23fd970123be735b9e14b7e6644cb'}, 'research_rag/evaluation/compare_confidence_backends.py': {'size': 10701, 'sha256': '0d383a92d02cec05743c3585058cd1fd7a4dbd5daab3730beee08b499eb5e873'}, 'research_rag/evaluation/compare_factchecker.py': {'size': 18763, 'sha256': 'ce7a74e56699a35d597493dd5152972a0cf40447aab6000774c7d76209574ae7'}, 'research_rag/evaluation/decomposition_probe_claims_v2.jsonl': {'size': 1711, 'sha256': '440030681bca80724ab4ee0e85e79ac70db7f1703f77587991000cb07b6bea13'}, 'research_rag/evaluation/evaluate_atomic_verifier.py': {'size': 35817, 'sha256': '286ef65e29340c3bbd17c8b69076fc681c2d366a077aec6006b4ab8a47404e60'}, 'research_rag/evaluation/factchecker_contract_cases.jsonl': {'size': 3533, 'sha256': '918a1a95cfed421aa8748d7eac29af74a03f93825c30817e002b2b2e16feb7e3'}, 'research_rag/graph.py': {'size': 32194, 'sha256': 'b2883b3b7c2ca75cda20ab9223fb8012591dcd3284dc2eca9595af083fc275df'}, 'research_rag/indexing.py': {'size': 5245, 'sha256': '2e02675e5094a278b4eb9d0e2eb0245389fda47ad0d9236ba69fef68f457cd14'}, 'research_rag/local.py': {'size': 17378, 'sha256': 'b363825e64d07a5f8c12c5b456b1e58e4b2c4297e93f48d493c5dec3ba8bbf5a'}, 'research_rag/observe.py': {'size': 2854, 'sha256': '46300c0d7f859228523efc2359d01092c48d7317b7509f5f4efb8fc6bbaae051'}, 'research_rag/planning.py': {'size': 9967, 'sha256': '680b25c3d049f7fa908ebaaed9b2a799a2430425eefd33ffb2b362e36a2d24f7'}, 'research_rag/preflight.py': {'size': 5027, 'sha256': 'bccd1755d5e23462f334703822d6fb185ca089bd62e2be9214bc0b9558fb5e39'}, 'research_rag/retrieval.py': {'size': 12796, 'sha256': 'f8fc0c3118c0feb7dcb845f6be2a3dc026f253679524cbdcb5c4ec9bc1f6c75b'}, 'research_rag/state.py': {'size': 1901, 'sha256': '0da55a10e4f5d1c33b4f06cad2e78553bfb190bc30c8b96b76a699e69a6fdc39'}, 'research_rag/tests/fixtures/atomic_cellfix2_binding_traces.json': {'size': 19180, 'sha256': '734d96996de0008940e908d9513b5e21dd331d64c6124209c83e0af651cdbf9f'}, 'research_rag/tests/fixtures/atomic_cellfix3_citation_traces.json': {'size': 4284, 'sha256': '62ea69491c48c58c7441023f8314b04b9d0ef63b88fbcf58504654d94b891d82'}, 'research_rag/tests/fixtures/atomic_cellfix4_metric_traces.json': {'size': 1651, 'sha256': '19f7e6de615f35b5a5bbf276aa0639930b9c35140cd9ba9b2ab2a0a92a28187d'}, 'research_rag/tests/fixtures/atomic_cellfix5_full39_assessments.json': {'size': 86350, 'sha256': '5000916b16563b7db290957c799e72267db0a4dddada612c9f74b46ebc4cb2db'}, 'research_rag/tests/fixtures/atomic_v8_v7_assessments.json': {'size': 4591, 'sha256': '577eb9dd475a800b80b8330510573c0b0c52b53e57e565c77af79e7b71675494'}, 'research_rag/tests/test_atomic_assessment_cellfix6.py': {'size': 16759, 'sha256': '17675cb98d6017de47c9d9bf4c275940bddca93f08e246a2ba06a125739f5cc0'}, 'research_rag/tests/test_atomic_assessment_v8.py': {'size': 16879, 'sha256': '93e900e49a79b22313d0c74c5b3d1e8ff492292a7ad2674f87b12fcf7dcd9db5'}, 'research_rag/tests/test_atomic_contract_v5.py': {'size': 10515, 'sha256': 'c4bec92fda42241f7c0ee03366dda8e4ed88049e1ac5b75a0780f9f910c0cbed'}, 'research_rag/tests/test_atomic_decoding.py': {'size': 8267, 'sha256': '6eabd3db1e06ffa61021811b7909b65277e6d4719cbf7d1f3e26d74c79e1745b'}, 'research_rag/tests/test_atomic_decomposition_v5.py': {'size': 8424, 'sha256': 'f5f2e053f0dbe3de38e0751aded20f22bdfa7c3822e4e7b669dd3eea2baf82d9'}, 'research_rag/tests/test_atomic_experiment.py': {'size': 18917, 'sha256': '321e4cd31f7a10ee1b1805ddbd644bcb3e0ad3da290d12eaab96b92e503628ae'}, 'research_rag/tests/test_atomic_metric_cellfix5.py': {'size': 13444, 'sha256': 'fc66c70e90689eb9816ba8a8a4c6fe028832520e9bde12993ff97d3ad0f994fb'}, 'research_rag/tests/test_atomic_metric_v6.py': {'size': 7270, 'sha256': '63433a23b9b07a309c2f88afd8bbe6eb6c552a344d69a9e97a25dedbee095477'}, 'research_rag/tests/test_atomic_metric_v7.py': {'size': 15499, 'sha256': '6d05e7f7502457cec1962b0959c731e13b123e678fee19c0fe156423eab29299'}, 'research_rag/tests/test_atomic_smoke_cellfix.py': {'size': 5569, 'sha256': '9b4efc67525f4fc905739590839fcb54c9db5b0c55964a429469b86b0a428949'}, 'research_rag/tests/test_atomic_smoke_cellfix2.py': {'size': 3574, 'sha256': '6c99a4ed08116f7bb5a72790a6830bcf4417fdf8168a404bc35fc51061c3f11a'}, 'research_rag/tests/test_atomic_smoke_cellfix3.py': {'size': 12622, 'sha256': 'f240bd059d220ba21cca5a02deb17e39302cec73c22c1a473825599a44002d7b'}, 'research_rag/tests/test_atomic_smoke_cellfix4.py': {'size': 9804, 'sha256': '51009a2b500059634383a6a1961d9caf8d9156762f06fb9524b705562d728c49'}, 'research_rag/tests/test_atomic_verifier.py': {'size': 26537, 'sha256': 'f9204bb90a198087d15337cabd99e3152a8ddd0c54ca63baf606ef21b1d2137a'}, 'research_rag/tests/test_summary_schema.py': {'size': 2852, 'sha256': '95f04375510bca135ab56a31c40b1e6b418e126d0b5f9059db8da60dd71b521c'}, 'research_rag/tests/test_summary_trace.py': {'size': 4974, 'sha256': '5285b7109213573ef157e3b8d67e44d6eb78bc9e8b7e3d18290e7cbad5c47538'}, 'train_agent3_summarizer/config.py': {'size': 5213, 'sha256': 'cf69370bc520b960cf144386793f44fac316c1c1ea20fd9bb3e566519c6863ea'}, 'train_agent3_summarizer/critic/__init__.py': {'size': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'train_agent3_summarizer/critic/prompts.py': {'size': 3000, 'sha256': '5ec5559e8c6542375feb0d6ba619dc8b64036ae748972f9a01b8dfcb3ea54c11'}, 'train_agent3_summarizer/critic/train_critic_qlora.py': {'size': 19528, 'sha256': 'abf7cdf00ad5c1c76814e2a9425cb19088690a35ccfad0593b31d2d0f5e123d2'}, 'train_agent3_summarizer/eval_generator_critic.py': {'size': 27382, 'sha256': '9f0867034b0ac472a89afd0d8a6aa41b3a00e13ba89c2defbb7376c985f59e2b'}, 'train_agent3_summarizer/generate_training_data.py': {'size': 18476, 'sha256': 'ac2ec01492e5a73bf99700b8f189d247b451d42203c1447397ac8075f9f024ae'}, 'train_agent3_summarizer/generator/__init__.py': {'size': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'train_agent3_summarizer/generator/prompts.py': {'size': 1944, 'sha256': '64bf3da00181c210452f86f24cf67b9c8acd9ea12c66e5d33eb2f026327e46fd'}, 'train_agent3_summarizer/generator/train_generator_qlora.py': {'size': 19436, 'sha256': '1262f59c2a3a6be7815f9c3dee434bdde047a5e8a6da9ade82c84a06c9c34d11'}, 'train_agent3_summarizer/llm_judge_eval.py': {'size': 6421, 'sha256': 'cc1cfc18b9f1135e54fe35dd3702ad1724d1e7ca5ebc4b4e98b8273edd37f413'}, 'train_agent3_summarizer/merge_lora_weights.py': {'size': 2558, 'sha256': '65dcb7ed6f2711c01fe9a98b2bf83225d22aacaf11219d7fc8bd92f8ca33af8b'}, 'train_agent3_summarizer/node.py': {'size': 15263, 'sha256': '43dff6e0ecccd9b9f7c6eb24d1ee73c0239c45a0c3553faf499a5e9fcfad5096'}, 'train_agent3_summarizer/validate_evidence_span.py': {'size': 5874, 'sha256': '54795f2bf992d07a32a70c8f19852d022f1a4706dbb0b180cf87840bb43f17bf'}, 'validation/atomic-cellfix6-v1/REVIEW.html': {'size': 80685, 'sha256': '4b1349854c2eaef4acfc9a494f5550fc60aa8d8d6d962d1220af0374679ac9f1'}, 'validation/atomic-cellfix6-v1/annotation-review.template.json': {'size': 9449, 'sha256': '8ce1b3c135240aae002729a607eddb5e4198dab7528800a3c3773f7a8a7f2058'}, 'validation/atomic-cellfix6-v1/authoring.json': {'size': 12427, 'sha256': '919c35353f283d394624cb0252fbce07b0f48868d9a7a1c54c444f32e9af8a07'}, 'validation/atomic-cellfix6-v1/cases.jsonl': {'size': 48392, 'sha256': '1df43fb4907b1e385c8ad04059a81644d5aa3e84dd121ab93ee0f2b079ea13f5'}, 'validation/atomic-cellfix6-v1/criteria.json': {'size': 1703, 'sha256': 'fd75cfde6c180c2d16e88219742a1b27a3c1aedaec65d2d09ccd10095854d4bf'}, 'validation/atomic-cellfix6-v1/inputs.jsonl': {'size': 25016, 'sha256': 'da73a462421d7ca850ffe5061c3fcd8a4bad35a81cb7b68bd865ab24a276b77a'}, 'validation/atomic-cellfix6-v1/manifest.json': {'size': 1521, 'sha256': '2220c18b708c967b7a33ee1bcd6a8771ce7aefe7e2b3753d8a60adcc3c080069'}, 'validation/atomic-cellfix6-v1/overlap-report.json': {'size': 14933, 'sha256': '7cb2c60f63778f6dc5dbc2683221b45d3648c4f97fb5e995c4a96fc659cb711e'}, 'validation/atomic-cellfix6-v1/review_gate.py': {'size': 5695, 'sha256': 'b2f7ce817af4152ae5c19ce8ddd76083a796297d9f088a92fa9a2e81d1dcd7c4'}, 'validation/atomic-cellfix6-v1/source-provenance.json': {'size': 1713, 'sha256': '7ae871ed72ce3f21748f5d62463fb7dfb8f7b302227355e62df1192fa5b2d043'}}
criteria_sha256 = 'b9d77823b5c8d6d2a56d4053667b8b53f6e059a1f5edb9e7e3a7578a50f34071'
def bundle():
    # This dedicated directory contains only diagnostics produced by these cells.
    entries = [{'path':str(p.relative_to(out)), 'size':p.stat().st_size,
                'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
               for p in sorted(out.rglob('*')) if p.is_file() and p.name != 'output-manifest.json']
    (out/'output-manifest.json').write_text(json.dumps({'run_id':run_id,'files':entries},indent=2))
    archive = out.with_suffix('.zip')
    pending = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(pending, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file(): z.write(p, str(p.relative_to(out)))
    pending.replace(archive)
    print('Diagnostic archive:', archive.name, flush=True)

def run(name, command, *, timeout=1800, child_env=None, required=True):
    assert name not in statuses, 'Preserve the previous stage; start a fresh run for retries'
    started = time.perf_counter()
    status = {'state':'running'}
    statuses[name] = status
    (out/'status.json').write_text(json.dumps(statuses,indent=2))
    try:
        with (out/(name+'.stdout.log')).open('x') as stdout, (out/(name+'.stderr.log')).open('x') as stderr:
            process = subprocess.Popen(command, cwd=root, env=child_env or env,
                                       stdout=stdout, stderr=stderr, start_new_session=True)
            try:
                while True:
                    try:
                        code = process.wait(timeout=15)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.perf_counter()-started
                        print(name, 'running', round(elapsed), 'seconds', flush=True)
                        if elapsed > timeout: raise TimeoutError(name)
            except BaseException:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                raise
        status.update(state='completed' if code == 0 else 'failed', exit_code=code)
    except BaseException as exc:
        status.update(state='interrupted', error_type=type(exc).__name__)
        raise
    finally:
        status['elapsed_ms'] = round((time.perf_counter()-started)*1000,2)
        (out/'status.json').write_text(json.dumps(statuses,indent=2))
        bundle()
    print(name, status, flush=True)
    if required and code:
        print((out / (name + '.stderr.log')).read_text()[-12000:], flush=True)
        raise RuntimeError(name + ' failed; inspect its saved stderr')
    return code

def download_archive():
    bundle()
    files.download(str(out.with_suffix('.zip')))

try:
    raw=base64.b64decode(payload,validate=True)
    assert hashlib.sha256(raw).hexdigest()==payload_sha256
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        assert archive.testzip() is None
        assert len(archive.namelist())==len(manifest) and set(archive.namelist())==set(manifest)
        for name,item in manifest.items():
            relative=PurePosixPath(name)
            assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name
            data=archive.read(name)
            assert len(data)==item['size'] and hashlib.sha256(data).hexdigest()==item['sha256'],name
            target=root/relative
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(data)
    assert hashlib.sha256((root/'research_rag/evaluation/atomic_verifier_criteria_v8.json').read_bytes()).hexdigest()==criteria_sha256
    (out/'input-manifest.json').write_text(json.dumps(manifest,indent=2))
    (out/'frozen-inputs.zip').write_bytes(raw)
    (out/'notebook-provenance.json').write_text(json.dumps({
        'candidate':'atomic-verifier-constrained-v8-cellfix6',
        'criteria_sha256':criteria_sha256,'payload_sha256':payload_sha256,
        'startup':'fresh embedded inputs; no previous session required',
        'model_data_budgets_and_gates_unchanged':True},indent=2))
    bundle()
except BaseException:
    download_archive()
    raise
print('Frozen cellfix6 ready. Run directory:',run_root)

def require_validation_review(upload=False):
    # Recheck actual bytes, not a stale Boolean left over from another cell.
    for name,item in manifest.items():
        data=(root/name).read_bytes()
        if len(data)!=item['size'] or hashlib.sha256(data).hexdigest()!=item['sha256']:
            raise ValueError('Frozen input changed: '+name)
    review_path=out/'validation-annotation-review.json'
    if upload and not review_path.exists():
        print('Upload annotation-review-cellfix6-v1.json exported after your direct annotation review.')
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise ValueError('Upload exactly one review JSON; do not upload a ZIP or old probe review.')
        review_path.write_bytes(next(iter(uploaded.values())))
    if not review_path.exists():
        raise ValueError('Validation annotations are pending human review. Complete the review before GPU work.')
    import importlib.util
    folder=root/'validation/atomic-cellfix6-v1'
    spec=importlib.util.spec_from_file_location('validation_gate',folder/'review_gate.py')
    gate=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(gate)
    raw=review_path.read_bytes()
    lock=gate.admit(folder,json.loads(raw))
    if lock['candidate_criteria_sha256']!=criteria_sha256:
        raise ValueError('Annotation lock is for a different candidate')
    lock['review_sha256']=hashlib.sha256(raw).hexdigest()
    gate.write_once(out/'validation-lock.json',gate.json_bytes(lock))
    return lock


**2 — Nhận review annotation, khóa validation, dựng GPU**

In [ ]:
try:
    require_validation_review(upload=True)
except BaseException:
    download_archive()
    raise
try:
    run('gpu-host',['nvidia-smi'])
    run('venv', [sys.executable, '-m', 'venv', '--without-pip', str(venv)])
    run('bootstrap', [sys.executable, '-m', 'pip', '--python', python, 'install', '--no-cache-dir', 'pip==26.2.1', 'wrapt==2.1.2'])
    run('torch', [python, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.10.0', 'torchvision==0.25.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
    run('runtime', [python, '-m', 'pip', 'install', '--no-cache-dir', '-r', 'requirements-orchestrator.txt', '-r', 'requirements-atomic-verifier.txt'])
    run('pip-check', [python, '-m', 'pip', 'check'])
    run('pip-freeze', [python, '-m', 'pip', 'freeze'])
except BaseException:
    download_archive()
    raise


**3 — Regression, ba validate-only, model cache**

In [ ]:
try:
    require_validation_review(upload=False)
except BaseException:
    download_archive()
    raise
try:
    run('regression',[python,'-m','unittest','research_rag.tests.test_atomic_verifier','research_rag.tests.test_atomic_experiment','research_rag.tests.test_atomic_decoding','research_rag.tests.test_atomic_decomposition_v5','research_rag.tests.test_atomic_contract_v5','research_rag.tests.test_summary_schema','research_rag.tests.test_summary_trace','research_rag.tests.test_atomic_metric_v6','research_rag.tests.test_atomic_metric_v7','research_rag.tests.test_atomic_assessment_v8','research_rag.tests.test_atomic_smoke_cellfix','research_rag.tests.test_atomic_smoke_cellfix2','research_rag.tests.test_atomic_smoke_cellfix3','research_rag.tests.test_atomic_smoke_cellfix4','research_rag.tests.test_atomic_metric_cellfix5','research_rag.tests.test_atomic_assessment_cellfix6','-v'],timeout=900)
    run('cuda-preflight',[python,'-c',"import json,torch; assert torch.cuda.is_available(); print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}))"])
    for name,flag in (('probe-validate-only','--decomposition-probe'),('smoke-validate-only','--assessment-smoke'),('verifier-validate-only',None)):
        command=[python,'-m','research_rag.evaluation.evaluate_atomic_verifier','--output-dir',str(out/'unused'),'--validate-only']
        if flag: command.append(flag)
        run(name,command)
    run('pinned-model-cache',[python,'-c',"from huggingface_hub import snapshot_download; print(snapshot_download('Qwen/Qwen2.5-3B-Instruct',revision='aa8e72537993ba99e69dfaafa59ed015b17504d1',ignore_patterns=['*.bin','*.gguf','original/*']))"],timeout=1800)
except BaseException:
    download_archive()
    raise
print('Regression, CUDA, freeze and pinned model-cache checks passed.')


**4 — Smoke12**

In [ ]:
try:
    require_validation_review(upload=False)
except BaseException:
    download_archive()
    raise
smoke_passed = False
try:
    assert all(statuses.get(n,{}).get('exit_code')==0 for n in ('gpu-host','venv','bootstrap','torch','runtime','pip-check','pip-freeze','regression','cuda-preflight','probe-validate-only','smoke-validate-only','verifier-validate-only','pinned-model-cache')), 'Complete cells 1-3 first.'
    code=run('assessment-smoke',[python,'-u','-m','research_rag.evaluation.evaluate_atomic_verifier',
        '--output-dir',str(out/'assessment-smoke'),'--assessment-smoke','--online-model-downloads'],
        timeout=2520,required=False)
    path=out/'assessment-smoke/metrics.json'
    metrics=json.loads(path.read_text()) if path.exists() else {}
    print(json.dumps(metrics,indent=2))
    smoke_passed=code==0 and metrics.get('automated_gate_pass') is True
finally:
    download_archive()
if smoke_passed:
    print('SMOKE DAT: co the chay probe18 o cell tiep theo. Full39 van can human review moi.')
else:
    print('SMOKE CHUA DAT: da luu ZIP; probe18/full39 bi chan. Cell ket thuc khong co nghia la gate da dat.')


**5 — Probe18 sau smoke**

In [ ]:
try:
    require_validation_review(upload=False)
except BaseException:
    download_archive()
    raise
if not globals().get('smoke_passed',False):
    print('BO QUA PROBE18: assessment-smoke chua dat. Giu ZIP chan doan va dung tai day.')
else:
    path=out/'assessment-smoke/metrics.json'
    confirmed=(statuses.get('assessment-smoke',{}).get('exit_code')==0 and path.exists()
        and json.loads(path.read_text()).get('automated_gate_pass') is True)
    if not confirmed:
        smoke_passed=False
        print('BO QUA PROBE18: khong co ket qua smoke dat cho run hien tai.')
    else:
        try:
            code=run('decomposition-probe',[python,'-u','-m','research_rag.evaluation.evaluate_atomic_verifier',
                '--output-dir',str(out/'decomposition-probe'),'--decomposition-probe','--online-model-downloads'],
                timeout=2520,required=False)
            for name in ('summary.json','metrics.json'):
                path=out/'decomposition-probe'/name
                print(name,path.read_text() if path.exists() else 'Incomplete; inspect saved logs.')
        finally:
            download_archive()
        print('Gui ZIP moi de audit va review 18 claim. Human review van pending; chua chay full39 hoac serving.')
